# Tutorial: Event Ingestion

Audience:
- Developers validating event provider ingestion behavior.

Prerequisites:
- Network/data provider access for configured ingestion providers.

Learning goals:
- Collect events for a symbol universe and time window.
- Group and inspect events by symbol.
- Test provider-specific fetch behavior.


## Outline

1. Setup imports and date range
2. Collect events across providers
3. Summarize by symbol
4. Test direct provider access


In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta

from swing_screener.intelligence.ingestion import collect_events
from swing_screener.intelligence.ingestion.factory import get_event_provider


## Step 1 - Define symbols and lookback range

Use a 72-hour window to match default catalyst lookback examples.


In [ ]:
symbols = ["NVDA", "AMD", "AAPL", "MSFT", "TSM"]
now = datetime.now()
start_dt = now - timedelta(hours=72)
end_dt = now

print(f"Collecting events for {symbols}")
print(f"Time range: {start_dt} to {end_dt}")


## Step 2 - Collect and summarize events

Group by symbol and show a couple of headlines per symbol.


In [ ]:
events = collect_events(
    symbols=symbols,
    start_dt=start_dt,
    end_dt=end_dt,
    provider_names=("yahoo_finance",),
)

print(f"Total events collected: {len(events)}")

by_symbol = defaultdict(list)
for event in events:
    by_symbol[event.symbol].append(event)

for symbol, evts in sorted(by_symbol.items()):
    print(f"{symbol}: {len(evts)} events")
    for evt in evts[:2]:
        print(f"  - {evt.headline[:60]}...")
        print(f"    type={evt.event_type}, credibility={evt.credibility:.2f}")


## Step 3 - Test provider fetch methods directly

Validate both Yahoo Finance and earnings-calendar providers.


In [ ]:
provider = get_event_provider("yahoo_finance")
events_yf = provider.fetch(symbols=symbols, start_dt=start_dt, end_dt=end_dt)
print(f"Yahoo Finance events: {len(events_yf)}")

try:
    earnings_provider = get_event_provider("earnings_calendar")
    events_earnings = earnings_provider.fetch(
        symbols=symbols,
        start_dt=start_dt,
        end_dt=end_dt,
    )
    print(f"Earnings calendar events: {len(events_earnings)}")
except Exception as e:
    print(f"Earnings calendar provider error: {e}")
